# Localization Ensemble Pipeline

Pipeline untuk object localization dengan OWL-ViT, feature extraction dengan pretrained model, dan ensemble traditional ML models.

## 1. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from transformers import OwlViTProcessor, OwlViTForObjectDetection
from transformers import AutoFeatureExtractor, AutoModel, AutoModelForImageClassification
from torch.utils.data import Dataset, DataLoader
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
import warnings
warnings.filterwarnings('ignore')

## 2. Configuration and Data Loading

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

train_df = pd.read_csv('train.csv')
train_dir = 'train/train/'
cropped_dir = 'cropped_localization/'
os.makedirs(cropped_dir, exist_ok=True)

print(f"Total samples: {len(train_df)}")
print(train_df.head())

## 3. Initialize OWL-ViT for Localization

In [ ]:
owl_processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
owl_model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32").to(device)
owl_model.eval()

text_queries = [["clothing", "shirt", "dress", "garment", "apparel", "jacket", 'hoodie', "tshirt"]]
print("OWL-ViT model loaded")

## 4. Initialize Feature Extraction Model

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/convnext-base-224")
feature_model = AutoModel.from_pretrained("facebook/convnext-base-224").to(device)
feature_model.eval()

print("Feature extraction model loaded")

## 5. Image Preprocessing Functions

In [ ]:
def load_image(image_id, train_dir):
    for ext in ['.jpg', '.png', '.jpeg']:
        image_path = os.path.join(train_dir, f"{image_id}{ext}")
        if os.path.exists(image_path):
            image = Image.open(image_path).convert('RGB')
            return image, image_path
    return None, None

def preprocess_image(image, target_size=(768, 768)):
    image_resized = image.resize(target_size, Image.LANCZOS)
    return image_resized

## 6. Object Localization and Cropping

In [ ]:
def detect_object(image, threshold=0.1):
    inputs = owl_processor(text=text_queries, images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = owl_model(**inputs)
    
    target_sizes = torch.tensor([image.size[::-1]]).to(device)
    results = owl_processor.post_process_object_detection(
        outputs=outputs, 
        threshold=threshold, 
        target_sizes=target_sizes
    )[0]
    
    boxes = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    
    return boxes, scores

def crop_object(image, boxes, padding=10):
    if len(boxes) == 0:
        return image
    
    image_np = np.array(image)
    height, width = image_np.shape[:2]
    
    best_box = boxes[0]
    x1, y1, x2, y2 = map(int, best_box)
    
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(width, x2 + padding)
    y2 = min(height, y2 + padding)
    
    cropped = image_np[y1:y2, x1:x2]
    
    return Image.fromarray(cropped)

## 7. Feature Extraction with EfficientNet

In [ ]:
def extract_features(image):
    inputs = feature_extractor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = feature_model(**inputs)
        
    if hasattr(outputs, 'pooler_output'):
        features = outputs.pooler_output
    elif hasattr(outputs, 'last_hidden_state'):
        features = outputs.last_hidden_state.mean(dim=1)
    else:
        features = outputs[0].mean(dim=[2, 3])
    
    return features.cpu().numpy().flatten()

## 8. Complete Pipeline Function

In [ ]:
def process_single_image(image_id):
    image, img_path = load_image(image_id, train_dir)
    if image is None:
        return None
    
    preprocessed = preprocess_image(image)
    boxes, scores = detect_object(preprocessed)
    cropped = crop_object(preprocessed, boxes, padding=10)
    features = extract_features(cropped)
    
    return features

## 9. Test OWL-ViT Cropping Pipeline

In [ ]:
import matplotlib.pyplot as plt

test_samples = [1, 5, 10, 20, 50, 100]
num_samples = min(6, len(test_samples))

fig, axes = plt.subplots(num_samples, 3, figsize=(15, num_samples * 4))
if num_samples == 1:
    axes = axes.reshape(1, -1)

success_count = 0
for idx, img_id in enumerate(test_samples[:num_samples]):
    image, img_path = load_image(img_id, train_dir)
    
    if image is not None:
        preprocessed = preprocess_image(image)
        boxes, scores = detect_object(preprocessed)
        cropped = crop_object(preprocessed, boxes, padding=10)
        
        axes[idx, 0].imshow(image)
        axes[idx, 0].set_title(f'Original - ID: {img_id}')
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(preprocessed)
        if len(boxes) > 0:
            for box, score in zip(boxes[:1], scores[:1]):
                x1, y1, x2, y2 = box
                rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                                    fill=False, color='red', linewidth=2)
                axes[idx, 1].add_patch(rect)
                axes[idx, 1].text(x1, y1-5, f'{score:.2f}', 
                                color='red', fontsize=10, 
                                bbox=dict(facecolor='white', alpha=0.7))
        axes[idx, 1].set_title(f'Detection ({len(boxes)} objects)')
        axes[idx, 1].axis('off')
        
        axes[idx, 2].imshow(cropped)
        axes[idx, 2].set_title(f'Cropped Object')
        axes[idx, 2].axis('off')
        
        success_count += 1

plt.tight_layout()
plt.show()

print(f"\nOWL-ViT cropping pipeline successful: {success_count}/{num_samples} images processed")

## 10. Extract Features from Dataset

In [ ]:
features_list = []
labels_jenis = []
labels_warna = []

print("Extracting features...")
for idx, row in train_df.iterrows():
    features = process_single_image(row['id'])
    
    if features is not None:
        features_list.append(features)
        labels_jenis.append(row['jenis'])
        labels_warna.append(row['warna'])
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(train_df)}")

X = np.array(features_list)
y_jenis = np.array(labels_jenis)
y_warna = np.array(labels_warna)

print(f"\nFeature shape: {X.shape}")
print(f"Jenis labels: {len(y_jenis)}")
print(f"Warna labels: {len(y_warna)}")

## 11. Pretrained Model for Warna Classification

In [ ]:
class ClothingDataset(Dataset):
    def __init__(self, df, transform, train_dir):
        self.df = df
        self.transform = transform
        self.train_dir = train_dir
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row['id']
        
        image, _ = load_image(image_id, self.train_dir)
        if image is None:
            image = Image.new('RGB', (224, 224), color='white')
        else:
            preprocessed = preprocess_image(image)
            boxes, scores = detect_object(preprocessed)
            image = crop_object(preprocessed, boxes, padding=10)
        
        image_tensor = self.transform(images=image, return_tensors="pt")['pixel_values'][0]
        label = row['warna']
        
        return image_tensor, label

warna_classifier = AutoModelForImageClassification.from_pretrained(
    "microsoft/resnet-50",
    num_labels=len(unique_warna),
    ignore_mismatched_sizes=True
).to(device)

print("Pretrained model for Warna loaded")

## 12. Prepare DataLoaders for Warna Training

In [ ]:
train_df_split, val_df_split = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['warna'])

train_dataset = ClothingDataset(train_df_split, feature_extractor, train_dir)
val_dataset = ClothingDataset(val_df_split, feature_extractor, train_dir)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

print(f"Train dataset: {len(train_dataset)}")
print(f"Val dataset: {len(val_dataset)}")

## 13. Fine-tune Pretrained Model for Warna

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(warna_classifier.parameters(), lr=2e-5)

num_epochs = 5
best_val_acc = 0.0

print("Training Warna classifier...")
for epoch in range(num_epochs):
    warna_classifier.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = warna_classifier(images).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
    
    train_acc = train_correct / train_total
    
    warna_classifier.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = warna_classifier(images).logits
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = val_correct / val_total
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
    
    print(f"Epoch {epoch+1}/{num_epochs} - Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")

## 14. Evaluate Pretrained Model on Test Set

In [ ]:
warna_classifier.eval()

y_true_warna_pretrained = []
y_pred_warna_pretrained = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = warna_classifier(images).logits
        _, predicted = outputs.max(1)
        
        y_true_warna_pretrained.extend(labels.cpu().numpy())
        y_pred_warna_pretrained.extend(predicted.cpu().numpy())

y_true_warna_pretrained = np.array(y_true_warna_pretrained)
y_pred_warna_pretrained = np.array(y_pred_warna_pretrained)

pretrained_warna_acc = accuracy_score(y_true_warna_pretrained, y_pred_warna_pretrained)

print(f"Pretrained Model Warna Accuracy: {pretrained_warna_acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_true_warna_pretrained, y_pred_warna_pretrained))

## 15. Confusion Matrix - Pretrained Warna Model

In [ ]:
cm_warna_pretrained = confusion_matrix(y_true_warna_pretrained, y_pred_warna_pretrained)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_warna_pretrained, annot=True, fmt='d', cmap='Greens',
            xticklabels=unique_warna, yticklabels=unique_warna)
plt.title(f'Confusion Matrix - Warna (Pretrained Model)\nAccuracy: {pretrained_warna_acc:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 16. Check Class Distribution (For Traditional ML)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

unique_jenis, counts_jenis = np.unique(y_jenis, return_counts=True)
unique_warna, counts_warna = np.unique(y_warna, return_counts=True)

print("Jenis Distribution:")
for cls, count in zip(unique_jenis, counts_jenis):
    percentage = (count / len(y_jenis)) * 100
    print(f"  Class {cls}: {count} samples ({percentage:.2f}%)")

print("\nWarna Distribution:")
for cls, count in zip(unique_warna, counts_warna):
    percentage = (count / len(y_warna)) * 100
    print(f"  Class {cls}: {count} samples ({percentage:.2f}%)")

class_weights_jenis = compute_class_weight('balanced', classes=unique_jenis, y=y_jenis)
class_weights_warna = compute_class_weight('balanced', classes=unique_warna, y=y_warna)

class_weight_dict_jenis = dict(zip(unique_jenis, class_weights_jenis))
class_weight_dict_warna = dict(zip(unique_warna, class_weights_warna))

print("\nClass Weights for Jenis:", class_weight_dict_jenis)
print("Class Weights for Warna:", class_weight_dict_warna)

## 17. Train-Test Split and Scaling (For Traditional ML - Jenis Only)

In [ ]:
X_train, X_test, y_jenis_train, y_jenis_test, y_warna_train, y_warna_test = train_test_split(
    X, y_jenis, y_warna, test_size=0.2, random_state=42, stratify=y_jenis
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")

## 18. Initialize Traditional ML Models for Jenis Classification

In [ ]:
models_jenis = {
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.1, random_state=42, eval_metric='mlogloss'),
    'HistGradient': HistGradientBoostingClassifier(max_iter=200, max_depth=7, learning_rate=0.1, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced'),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, class_weight='balanced'),
    'Ridge': RidgeClassifier(alpha=1.0, random_state=42, class_weight='balanced')
}

print("Models initialized for Jenis classification")

## 19. Train Traditional ML Models for Jenis Classification

## 20. Comparison - Traditional ML vs Pretrained Model for Warna

In [ ]:
print("="*60)
print("COMPARISON: Traditional ML vs Pretrained Model for Warna")
print("="*60)

print(f"\nPretrained Model (ResNet-50): {pretrained_warna_acc:.4f}")

print("\nNote: Using Pretrained Model for Warna Classification")
print("For Jenis Classification, continuing with Traditional ML models...")

## 21. Ensemble Model for Jenis Classification

In [ ]:
estimators_jenis = [(name, model) for name, model in trained_models_jenis.items()]
ensemble_jenis = VotingClassifier(estimators=estimators_jenis, voting='hard')
ensemble_jenis.fit(X_train_scaled, y_jenis_train)
y_pred_jenis = ensemble_jenis.predict(X_test_scaled)
acc_jenis = accuracy_score(y_jenis_test, y_pred_jenis)

print(f"Ensemble Jenis Accuracy: {acc_jenis:.4f}")
print(f"Pretrained Model Warna Accuracy: {pretrained_warna_acc:.4f}")

## 22. Final Classification Reports

In [ ]:
print("="*50)
print("JENIS CLASSIFICATION REPORT (Ensemble)")
print("="*50)
print(classification_report(y_jenis_test, y_pred_jenis))

print("\n" + "="*50)
print("WARNA CLASSIFICATION REPORT (Pretrained Model)")
print("="*50)
print(classification_report(y_true_warna_pretrained, y_pred_warna_pretrained))

## 23. Model Comparison Summary

## 24. Final Confusion Matrix Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cm_jenis = confusion_matrix(y_jenis_test, y_pred_jenis)
sns.heatmap(cm_jenis, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=unique_jenis, yticklabels=unique_jenis)
axes[0].set_title(f'Confusion Matrix - Jenis (Ensemble)\nAccuracy: {acc_jenis:.4f}')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

cm_warna = confusion_matrix(y_true_warna_pretrained, y_pred_warna_pretrained)
sns.heatmap(cm_warna, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=unique_warna, yticklabels=unique_warna)
axes[1].set_title(f'Confusion Matrix - Warna (Pretrained)\nAccuracy: {pretrained_warna_acc:.4f}')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    'Model_Jenis': list(results_jenis.keys()) + ['Ensemble'],
    'Jenis_Accuracy': list(results_jenis.values()) + [acc_jenis]
})

comparison_df = comparison_df.sort_values('Jenis_Accuracy', ascending=False)
print("JENIS CLASSIFICATION - Model Comparison:")
print(comparison_df.to_string(index=False))

print(f"\n\nWARNA CLASSIFICATION:")
print(f"Pretrained Model (ResNet-50): {pretrained_warna_acc:.4f}")

## 25. Save Best Models

In [ ]:
import pickle

models_dir = 'saved_models/'
os.makedirs(models_dir, exist_ok=True)

with open(f'{models_dir}ensemble_jenis.pkl', 'wb') as f:
    pickle.dump(ensemble_jenis, f)

torch.save(warna_classifier.state_dict(), f'{models_dir}warna_pretrained.pth')

with open(f'{models_dir}scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Models saved successfully")
print(f"- Jenis: Ensemble model (Traditional ML)")
print(f"- Warna: Pretrained ResNet-50 model")